# IA Generativa - Agente com Google ADK

Notebook organizado a partir do documento `docs/IA-UA09-Lab3-Python-Agent ADK Google`.

As dependencias ja foram instaladas com `uv add`; aqui fazemos apenas as importacoes, carregamos as variaveis de ambiente e reproduzimos os exemplos de agentes do laboratorio.

## 1. Configuracao de ambiente

Preencha o arquivo `.env` com as chaves reais antes de executar chamadas ao modelo.

In [1]:
import datetime
import os
from pathlib import Path
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from google.adk.agents import Agent
from groq import Groq

load_dotenv(dotenv_path=Path(".env"))

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "0")

google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

print("GOOGLE_API_KEY configurada:", bool(google_api_key))
print("GROQ_API_KEY configurada:", bool(groq_api_key))

GOOGLE_API_KEY configurada: True
GROQ_API_KEY configurada: True


In [2]:
# Cliente Groq opcional, caso a chave esteja configurada.
groq_client = Groq(api_key=groq_api_key) if groq_api_key else None
groq_client

## 2. Primeiro agente

Exemplo inicial do documento: um agente ADK com uma tool simples de horario.

In [3]:
def get_current_time_mock(city: str) -> dict:
    """Returns the current time in a specified city."""
    return {"status": "success", "city": city, "time": "10:30 AM"}


root_agent_simple = Agent(
    model="gemini-2.5-flash",
    name="root_agent_simple",
    description="A helpful assistant for user questions.",
    instruction="Answer user questions to the best of your knowledge",
    tools=[get_current_time_mock],
)

root_agent_simple

LlmAgent(name='root_agent_simple', description='A helpful assistant for user questions.', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=None, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model='gemini-2.5-flash', instruction='Answer user questions to the best of your knowledge', global_instruction='', static_instruction=None, tools=[<function get_current_time_mock at 0x0000022A03398AE0>], generate_content_config=None, mode=None, parallel_worker=None, disallow_transfer_to_parent=False, disallow_transfer_to_peers=False, include_contents='default', output_key=None, planner=None, code_executor=None, before_model_callback=None, after_model_callback=None, on_model_error_callback=None, before_tool_callback=None, after_tool_callback=None, on_tool_error_callback=None)

In [4]:
get_current_time_mock("New York")

{'status': 'success', 'city': 'New York', 'time': '10:30 AM'}

## 3. Segundo agente

Versao melhorada do agente, com tools para clima e horario.

In [5]:
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city: The name of the city for which to retrieve the weather report.

    Returns:
        A dictionary with status and result or error message.
    """
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": (
                "The weather in New York is sunny with a temperature of 25 degrees "
                "Celsius (77 degrees Fahrenheit)."
            ),
        }

    return {
        "status": "error",
        "error_message": f"Weather information for '{city}' is not available.",
    }


def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city.

    Args:
        city: The name of the city for which to retrieve the current time.

    Returns:
        A dictionary with status and result or error message.
    """
    if city.lower() == "new york":
        tz_identifier = "America/New_York"
    else:
        return {
            "status": "error",
            "error_message": f"Sorry, I don't have timezone information for {city}.",
        }

    tz = ZoneInfo(tz_identifier)
    now = datetime.datetime.now(tz)
    report = f'The current time in {city} is {now.strftime("%Y-%m-%d %H:%M:%S %Z%z")}'
    return {"status": "success", "report": report}

In [6]:
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction=(
        "You are a helpful agent who can answer user questions about the time "
        "and weather in a city."
    ),
    tools=[get_weather, get_current_time],
)

root_agent

LlmAgent(name='weather_time_agent', description='Agent to answer questions about the time and weather in a city.', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=None, output_schema=None, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model='gemini-2.0-flash', instruction='You are a helpful agent who can answer user questions about the time and weather in a city.', global_instruction='', static_instruction=None, tools=[<function get_weather at 0x0000022A03399EE0>, <function get_current_time at 0x0000022A0339A700>], generate_content_config=None, mode=None, parallel_worker=None, disallow_transfer_to_parent=False, disallow_transfer_to_peers=False, include_contents='default', output_key=None, planner=None, code_executor=None, before_model_callback=None, after_model_callback=None, on_model_error_callback=None, before_tool_callback=None, after_tool_callback=None, on_tool_error_callback=N

In [7]:
print(get_current_time("New York"))
print(get_current_time("Sao Jose dos Campos"))
print(get_weather("New York"))
print(get_weather("Sao Jose dos Campos"))

{'status': 'success', 'report': 'The current time in New York is 2026-06-02 04:40:57 EDT-0400'}
{'status': 'error', 'error_message': "Sorry, I don't have timezone information for Sao Jose dos Campos."}
{'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25 degrees Celsius (77 degrees Fahrenheit).'}
{'status': 'error', 'error_message': "Weather information for 'Sao Jose dos Campos' is not available."}


## 4. Execucao pelo ADK

Para executar pela CLI, salve uma estrutura de agente contendo `__init__.py`, `agent.py` e `.env`, ou use o comando do documento para gerar a estrutura base:

```bash
adk create my_agent
adk run my_agent
adk web --port 8000
```

Na interface web, teste perguntas como:

- Que horas sao em New York?
- Que horas sao em Sao Jose dos Campos?
- Como esta o tempo em New York?
- Como esta o clima em Sao Jose dos Campos?